 # Lab 1. Transformer Basics: Tokenization & Text Generation

   [![View notebooks on Github](https://img.shields.io/static/v1.svg?logo=github&label=Repo&message=View%20On%20Github&color=lightgrey)](https://github.com/cltl/nlp_transformer_tutorials/blob/main/NLP_Lab1_Transformer_Basics.ipynb)
 [![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cltl/nlp_transformer_tutorials/blob/main/NLP_Lab1_Transformer_Basics.ipynb)

 - This notebook is adapted from the [ARENA](https://www.arena.education/) research program, which provides excellent tutorial notebooks and walkthroughs for several topics related to alignment and the safety of AI. Since their explanations are very good, we won't try to redo theirs, but we adapt their setup, as they go a bit more in-depth on parts we will leave out for now. We have also restructured the notebooks and made other modifications to be a more standalone exercise.
 - These notebooks walk us through the Transformer architecture as sketched in [this excellent post](https://transformer-circuits.pub/2021/framework/index.html) on the transformer. The main point is that seeing the transformer as a residual network, and separating it into the individual components, gives us a lot more power to understand them!

 This is the first of three lab notebooks on transformers. In this notebook, we explore what a transformer does at a high level: how text is tokenized into integers, how those integers become vectors, what the model's output (logits) represents, and how we can use it to generate text autoregressively. We focus on GPT-2 as our running example.

 In the next notebook (*Lab 2: Building a Transformer from Scratch*), we will implement every component of the transformer architecture ourselves (embeddings, attention, MLPs, and the full model) and train it on a real dataset.

 If you prefer to get familiar with the technical details via a video, we recommend watching this [youtube video](https://youtu.be/eMlx5fFNoYc?si=P4zsYydKyLPc0HIY).

**Note for Running on Colab** If you're running this in Google Colab, make sure to use the GPU.
- Click the arrow on the right top, next to "RAM and DISK"
- Select "Change Runtime Type" -> Click `T4 GPU`

 ## Content & Learning Objectives

 > ##### Learning Objectives
 >
 > - Understand what a transformer is used for
 > - Understand causal attention, and what a transformer's output represents
 > - Learn what tokenization is, and how models do it
 > - Understand what logits are, and how to use them to derive a probability distribution over the vocabulary
 > - Walk through the full text generation pipeline step by step


### Install Env

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Clone the course repo (only needed in Colab)
if IN_COLAB:
    repo_url = "https://github.com/cltl/nlp_transformer_tutorials.git"
    repo_dir = "/content/nlp_transformer_tutorials"
    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    os.chdir(repo_dir)

# Make sure the repo root is on the Python path so `import utils.tests_part2` works
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

### Load Packages

In [ ]:
import os
import sys
import torch as t
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt


device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

 ## What is the point of a transformer?

 **Transformers exist to model text!**

 We're going to focus GPT-2 style transformers. Key feature: They generate text! You feed in language, and the model generates a probability distribution over tokens. And you can repeatedly sample from this to generate text!

 (To explain this in more detail - you feed in a sequence of length $N$, then sample from the probability distribution over the $N+1$-th word, use this to construct a new sequence of length $N+1$, then feed this new sequence into the model to get a probability distribution over the $N+2$-th word, and so on.)

 ### How is the model trained?

 You give it a bunch of text, and train it to predict the next token.

 Importantly, if you give a model 100 tokens in a sequence, it predicts the next token for *each* prefix, i.e. it produces 100 logit vectors (= probability distributions) over the set of all words in our vocabulary, with the `i`-th logit vector representing the probability distribution over the token *following* the `i`-th token in the sequence. This is a key part of what allows transformers to be trained so efficiently; for every sequence of length $n$ we get $n$ different predictions to train on:

 $$
 p(x_1), \; p(x_2|x_1), \; p(x_3|x_1x_2), \; \ldots, \; p(x_n|x_1 \ldots x_{n-1})
 $$

 <details>
 <summary>Aside - logits</summary>

 If you haven't encountered the term "logits" before, here's a quick refresher.

 Given an arbitrary vector $x$, we can turn it into a probability distribution via the **softmax** function: $x_i \to \frac{e^{x_i}}{\sum e^{x_j}}$. The exponential makes everything positive; the normalization makes it add to one.

 The model's output is the vector $x$ (one for each prediction it makes). We call this vector a logit because it represents a probability distribution, and it is related to the actual probabilities via the softmax function.
 </details>

 How do we stop the transformer by "cheating" by just looking at the tokens it's trying to predict? Answer - we make the transformer have *causal attention* (as opposed to *bidirectional attention*). Causal attention only allows information to move forwards in the sequence, never backwards. The prediction of what comes after token 50 is only a function of the first 50 tokens, *not* of token 51. We say the transformer is **autoregressive**, because it only predicts future words based on past data.

 <img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/transformer-overview-new.png" width="900">

 Another way to view this is through the following analogy: we have a series of people standing in a line, each with one word or chunk of the sentence. Each person has the ability to look up information from the people behind them (we'll explore how this works in later sections) but they can't look at any information in front of them. Their goal is to guess what word the person in front of them is holding.

 <img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/intro-image-v2.png" width="600">

 ## Tokens - Transformer Inputs

 Our transformer's input is natural language (i.e. a sequence of characters, strings, etc). But ML models generally take vectors as input, not language. How do we convert language to vectors?

 We can factor this into 2 questions:

 1. How do we split up language into small sub-units?
 2. How do we convert these sub-units into vectors?

 Let's start with the second of these questions.

 ### Converting sub-units to vectors

 We basically make a massive lookup table, which is called an **embedding**. It has one vector for each possible sub-unit of language we might get (we call this set of all sub-units our **vocabulary**). We label every element in our vocabulary with an integer (this labelling never changes), and we use this integer to index into the embedding.

 A key intuition is that one-hot encodings let you think about each integer independently. We don't bake in any relation between words when we perform our embedding, because every word has a completely separate embedding vector.

 <details>
 <summary>Aside - one-hot encodings</summary>

 We sometimes think about **one-hot encodings** of words. These are vectors with zeros everywhere, except for a single one in the position corresponding to the word's index in the vocabulary. This means that indexing into the embedding is equivalent to multiplying the **embedding matrix** by the one-hot encoding (where the embedding matrix is the matrix we get by stacking all the embedding vectors on top of each other).

 $$
 \begin{aligned}
 W_E &= \begin{bmatrix}
 \leftarrow v_0 \rightarrow \\
 \leftarrow v_1 \rightarrow \\
 \vdots \\
 \leftarrow v_{d_{vocab}-1} \rightarrow \\
 \end{bmatrix} \quad \text{is the embedding matrix (size }d_{vocab} \times d_{embed}\text{),} \\
 \\
 t_i &= (0, \dots, 0, 1, 0, \dots, 0) \quad \text{is the one-hot encoding for the }i\text{th word (length }d_{vocab}\text{)} \\
 \\
 v_i &= t_i W_E \quad \text{is the embedding vector for the }i\text{th word (length }d_{embed}\text{).} \\
 \end{aligned}
 $$

 </details>

 Now, let's answer the first question - how do we split language into sub-units?

 ### Splitting language into sub-units

 We need to define a standard way of splitting up language into a series of substrings, where each substring is a member of our **vocabulary** set.

 Could we use a dictionary, and have our vocabulary be the set of all words in the dictionary? No, because this couldn't handle arbitrary text (e.g. URLs, punctuation, etc). We need a more general way of splitting up language.

 Could we just use the 256 ASCII characters? This fixes the previous problem, but it loses structure of language - some sequences of characters are more meaningful than others. For example, "language" is a lot more meaningful than "hjksdfiu". We want "language" to be a single token, but not "hjksdfiu" - this is a more efficient use of our vocab.

 What actually happens? The most common strategy is called **Byte-Pair encodings**.

 We begin with the 256 ASCII characters as our tokens, and then find the most common pair of tokens, and merge that into a new token. Note that we do have a space character as one of our 256 tokens, and merges using space are very common. For instance, here are the five first merges for the tokenizer used by GPT-2 (you'll be able to verify this below).

 ```
 " t"
 " a"
 "he"
 "in"
 "re"
 ```

 Note - you might see the character `Ġ` in front of some tokens. This is a special token that indicates that the token begins with a space. Tokens with a leading space vs not are different.


 ### TransformerLens

 In this course we use [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens), a library built for mechanistic interpretability research. Unlike the standard HuggingFace `transformers` library (which is designed for training and inference), TransformerLens gives us direct access to every intermediate activation inside the model (attention patterns, residual stream values, MLP outputs, etc.) via a clean hook-based API. It also provides convenient utilities like `to_tokens`, `to_str_tokens`, and `run_with_cache` that we'll use extensively.

 We load models using `HookedTransformer.from_pretrained(...)` instead of the HuggingFace `AutoModel`. Let's load GPT-2 small and explore its tokenizer vocabulary:

In [3]:
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,  # these arguments relate to how TransformerLens handles layer normalization — not important for now
)

sorted_vocab = sorted(list(reference_gpt2.tokenizer.vocab.items()), key=lambda n: n[1])

print(sorted_vocab[:20])
print()
print(sorted_vocab[250:270])
print()
print(sorted_vocab[990:1010])
print()

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
[('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9), ('+', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), ('0', 15), ('1', 16), ('2', 17), ('3', 18), ('4', 19)]

[('ľ', 250), ('Ŀ', 251), ('ŀ', 252), ('Ł', 253), ('ł', 254), ('Ń', 255), ('Ġt', 256), ('Ġa', 257), ('he', 258), ('in', 259), ('re', 260), ('on', 261), ('Ġthe', 262), ('er', 263), ('Ġs', 264), ('at', 265), ('Ġw', 266), ('Ġo', 267), ('en', 268), ('Ġc', 269)]

[('Ġprodu', 990), ('Ġstill', 991), ('led', 992), ('ah', 993), ('Ġhere', 994), ('Ġworld', 995), ('Ġthough', 996), ('Ġnum', 997), ('arch', 998), ('imes', 999), ('ale', 1000), ('ĠSe', 1001), ('ĠIf', 1002), ('//', 1003), ('ĠLe', 1004), ('Ġret', 1005), ('Ġref', 1006), ('Ġtrans', 1007), ('ner', 1008), ('ution', 1009)]



 As you get to the end of the vocabulary, you'll be producing some pretty weird-looking esoteric tokens (because you'll already have exhausted all of the short frequently-occurring ones):

In [4]:
print(sorted_vocab[-20:])

[('Revolution', 50237), ('Ġsnipers', 50238), ('Ġreverted', 50239), ('Ġconglomerate', 50240), ('Terry', 50241), ('794', 50242), ('Ġharsher', 50243), ('Ġdesolate', 50244), ('ĠHitman', 50245), ('Commission', 50246), ('Ġ(/', 50247), ('âĢ¦."', 50248), ('Compar', 50249), ('Ġamplification', 50250), ('ominated', 50251), ('Ġregress', 50252), ('ĠCollider', 50253), ('Ġinformants', 50254), ('Ġgazed', 50255), ('<|endoftext|>', 50256)]


 <details>
 <summary>Fun (optional) exercise - can you guess what the first-formed 3/4/5/6/7-letter encodings in GPT-2's vocabulary are?</summary>
 Run this code to find out:

 ```python
 lengths = dict.fromkeys(range(3, 8), "")
 for tok, idx in sorted_vocab:
     if not lengths.get(len(tok), True):
         lengths[len(tok)] = tok

 for length, tok in lengths.items():
     print(f"{length}: {tok}")
 ```
 </details>

 Transformers in the `transformer_lens` library have a `to_tokens` method that converts text to numbers. It also prepends them with a special token called BOS (beginning of sequence) to indicate the start of a sequence. You can disable this with the `prepend_bos=False` argument.

 <details>
 <summary>Aside - BOS token</summary>

 The beginning of sequence (BOS) token is a special token used to mark the beginning of the sequence. Confusingly, in GPT-2, the End of Sequence (EOS), Beginning of Sequence (BOS) and Padding (PAD) tokens are all the same, `<|endoftext|>` with index `50256`.

 Why is this token added? Some basic intuitions are:

 * It provides context that this is the start of a sequence, which can help the model generate more appropriate text.
 * It can act as a "rest position" for attention heads (more on this later, when we discuss attention).

 TransformerLens adds this token automatically (including in forward passes of transformer models, e.g. it's implicitly added when you call `model("Hello World")`). You can disable this behaviour by setting the flag `prepend_bos=False` in `to_tokens`, `to_str_tokens`, `model.forward` and any other function that converts strings to multi-token tensors.

 **Key Point: *If you get weird off-by-one errors, check whether there's an unexpected `prepend_bos`!***

 Why are the BOS, EOS and PAD tokens the same? This is because GPT-2 is an autoregressive model, and uses these kinds of tokens in a slightly different way to other transformer families (e.g. BERT). For instance, GPT has no need to distinguish between BOS and EOS tokens, because it only processes text from left to right.

 </details>

 ### Some tokenization annoyances

 There are a few funky and frustrating things about tokenization, which causes it to behave differently than you might expect. For instance:

 #### Whether a word begins with a capital or space matters!

### Tokenization Exploration (Exercise 1)

> ```yaml
> Difficulty: easy
> Time: ~5 minutes
> ```

We just said that capitalization and leading spaces affect tokenization. Try it out by filling in your name below and see how the four variants get tokenized differently.

In [5]:
# Exercise 1: Fill in your name and run the cell
my_word = "..."  # TODO: fill in your name here! (lowercase, no spaces)

print("word:          ", reference_gpt2.to_str_tokens(my_word))
print("space + word:  ", reference_gpt2.to_str_tokens(" " + my_word))
print("Capitalized:   ", reference_gpt2.to_str_tokens(my_word.capitalize()))
print("space + Cap:   ", reference_gpt2.to_str_tokens(" " + my_word.capitalize()))

word:           ['<|endoftext|>', '...']
space + word:   ['<|endoftext|>', ' ...']
Capitalized:    ['<|endoftext|>', '...']
space + Cap:    ['<|endoftext|>', ' ...']


What do you see? Which variants produce different numbers of tokens, and why do you think that is? Discuss with your neighbour or TA.

 #### Arithmetic is a mess.

 Length is inconsistent, common numbers bundle together.

In [6]:
print(reference_gpt2.to_str_tokens("56873+3184623=123456789-1000000000"))

['<|endoftext|>', '568', '73', '+', '318', '46', '23', '=', '123', '45', '67', '89', '-', '1', '000000', '000']


 > ### Key Takeaways
 >
 > * We learn a dictionary of vocab of tokens (sub-words).
 > * We (approx) losslessly convert language to integers via tokenizing it.
 > * We convert integers to vectors via a lookup table.
 > * Note: input to the transformer is a sequence of *tokens* (ie integers), not vectors

 ## Text generation

 Now that we understand the basic ideas here, let's go through the entire process of text generation, from our original string to a new token which we can append to our string and plug back into the model.

 #### **Step 1:** Convert text to tokens

 The sequence gets tokenized, so it has shape `[batch, seq_len]`. Here, the batch dimension is just one (because we only have one sequence).

In [7]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will be able to generate coherent and interesting text!"
tokens = reference_gpt2.to_tokens(reference_text).to(device)
print(tokens)
print(tokens.shape)
print(reference_gpt2.to_str_tokens(tokens))


tensor([[50256,    40,   716,   281,  4998,  1960,   382, 19741,    11,   875,
         12342,    12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,
            13,  1881,  1110,   314,   481,   307,  1498,   284,  7716, 24870,
           290,  3499,  2420,     0]], device='cuda:0')
torch.Size([1, 34])
['<|endoftext|>', 'I', ' am', ' an', ' amazing', ' aut', 'ore', 'gressive', ',', ' dec', 'oder', '-', 'only', ',', ' G', 'PT', '-', '2', ' style', ' transformer', '.', ' One', ' day', ' I', ' will', ' be', ' able', ' to', ' generate', ' coherent', ' and', ' interesting', ' text', '!']


 #### **Step 2:** Map tokens to logits


 From our input of shape `[batch, seq_len]`, we get output of shape `[batch, seq_len, vocab_size]`. The `[i, j, :]`-th element of our output is a vector of logits representing our prediction for the `j+1`-th token in the `i`-th sequence.

In [8]:
logits, cache = reference_gpt2.run_with_cache(tokens)
print(logits.shape)

torch.Size([1, 34, 50257])


 (`run_with_cache` tells the model to cache all intermediate activations. This isn't important right now; we'll use it extensively in Lab 2 when we inspect the model's internals.)

 #### **Step 3:** Convert the logits to a distribution with a softmax

 This doesn't change the shape, it is still `[batch, seq_len, vocab_size]`.

In [9]:
probs = logits.softmax(dim=-1)
print(probs.shape)

torch.Size([1, 34, 50257])


### Implement Softmax (Exercise 2)

> ```yaml
> Difficulty: easy
> Time: ~5-10 minutes
> ```

We just used PyTorch's [`.softmax()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html) -- now let's implement it from scratch.

Recall that softmax converts a vector of logits $\mathbf{z} = (z_1, z_2, \ldots, z_n)$ into a probability distribution:

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}
$$

The exponential ([`t.exp`](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)) makes everything positive, and dividing by the sum makes it add up to 1.

One practical issue: $e^{z_i}$ overflows for large $z_i$. The fix is to subtract the max first:

$$
\text{softmax}(z_i) = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j=1}^{n} e^{z_j - \max(\mathbf{z})}}
$$

This is mathematically identical (the $e^{-\max(\mathbf{z})}$ cancels), but keeps the numbers manageable.

So the three steps are:
1. Subtract the max along the target dimension
2. Exponentiate
3. Divide by the sum

In [ ]:
def my_softmax(logits: t.Tensor, dim: int = -1) -> t.Tensor:
    """Implement softmax from scratch."""
    # TODO: Subtract the max for numerical stability
    # TODO: Exponentiate
    # TODO: Normalize by the sum
    raise NotImplementedError()

# Test against PyTorch's softmax
my_probs = my_softmax(logits, dim=-1)
assert t.allclose(my_probs, probs, atol=1e-5), "Your softmax doesn't match PyTorch's!"
print("Softmax test passed!")
print(f"Sum of probabilities for last position: {my_probs[0, -1].sum().item():.6f} (should be 1.0)")

In [ ]:
# Grab the probability distribution for the last token position
last_probs = my_probs[0, -1].cpu()

# Sort in descending order and plot the top 30 tokens
sorted_probs, sorted_indices = last_probs.sort(descending=True)
top_n = 30
top_tokens = [reference_gpt2.to_string(idx) for idx in sorted_indices[:top_n]]

plt.figure(figsize=(10, 4))
plt.bar(range(top_n), sorted_probs[:top_n].detach().numpy())
plt.xticks(range(top_n), top_tokens, rotation=60, ha="right", fontsize=8)
plt.ylabel("Probability")
plt.title("Softmax output: top 30 most likely next tokens")
plt.tight_layout()
plt.show()

# Show how peaked the distribution is
print(f"Top token gets {sorted_probs[0].item()*100:.1f}% of the probability")
print(f"Top 10 tokens together get {sorted_probs[:10].sum().item()*100:.1f}%")
print(f"The other {len(last_probs) - 10} tokens share the remaining {sorted_probs[10:].sum().item()*100:.1f}%")

<details><summary>Hint</summary>

Use `keepdim=True` in both `max()` and `sum()` so that broadcasting works:
```python
shifted = logits - logits.max(dim=dim, keepdim=True).values
exp_vals = t.exp(shifted)
return exp_vals / exp_vals.sum(dim=dim, keepdim=True)
```
</details>

 #### **Bonus step:** What is the most likely next token at each position?

In [12]:
most_likely_next_tokens = reference_gpt2.tokenizer.batch_decode(logits.argmax(dim=-1)[0])

print(list(zip(reference_gpt2.to_str_tokens(tokens), most_likely_next_tokens)))


[('<|endoftext|>', '\n'), ('I', "'m"), (' am', ' a'), (' an', ' avid'), (' amazing', ' person'), (' aut', 'od'), ('ore', 'sp'), ('gressive', '.'), (',', ' and'), (' dec', 'ently'), ('oder', ','), ('-', 'driven'), ('only', ' programmer'), (',', ' and'), (' G', 'IM'), ('PT', '-'), ('-', 'only'), ('2', '.'), (' style', ','), (' transformer', '.'), ('.', ' I'), (' One', ' of'), (' day', ' I'), (' I', ' will'), (' will', ' be'), (' be', ' able'), (' able', ' to'), (' to', ' use'), (' generate', ' a'), (' coherent', ','), (' and', ' coherent'), (' interesting', ' data'), (' text', ' in'), ('!', '\n')]


 We can see that, in a few cases (particularly near the end of the sequence), the model accurately predicts the next token in the sequence. We might guess that common phrases the model has seen in training are easier for it to predict.

### Top-K Predictions (Exercise 3)

> ```yaml
> Difficulty: easy
> Time: ~10 minutes
> ```

The bonus step above shows only the single best prediction per position. But the model outputs a full distribution over 50,257 tokens -- let's look at the top 5.

Write a function that takes an input string, runs it through the model, and prints the 5 most likely next tokens with their probabilities. You'll need:
- `reference_gpt2.to_tokens(text)` to tokenize
- `reference_gpt2(tokens)` to get logits
- `logits[0, -1]` to grab the last position
- `.softmax(dim=-1)` to get probabilities
- [`t.topk(tensor, k)`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html) which returns `.values` and `.indices`

In [ ]:
def show_top_k_predictions(text: str, k: int = 5):
    """Show the top-k most likely next tokens for the given text."""
    tokens = ...   # TODO: tokenize
    logits = ...   # TODO: run model
    last_logits = ...   # TODO: last position only
    probs = ...   # TODO: softmax
    top_probs, top_indices = ...   # TODO: top-k

    # Print results
    print(f"Input: {text!r}")
    print(f"Top {k} predictions for next token:")
    for i in range(k):
        token_str = reference_gpt2.to_string(top_indices[i])
        prob = top_probs[i].item() * 100
        print(f"  {i+1}. {token_str!r:15s} ({prob:.1f}%)")

# Try it!
show_top_k_predictions("The capital of France is")
print()
show_top_k_predictions("1 + 1 =")

<details><summary>Hint</summary>

Same indexing pattern as Step 4: `logits[0, -1]` gets the prediction for the next token. Then `t.topk(probs, k)` returns values sorted highest to lowest.
</details>

 #### **Step 4:** Map distribution to a token

In [13]:
next_token = logits[0, -1].argmax(dim=-1)
next_char = reference_gpt2.to_string(next_token)
print(repr(next_char))


'\n'


 Note that we're indexing `logits[0, -1]`. This is because logits have shape `[1, sequence_length, vocab_size]`, so this indexing returns the vector of length `vocab_size` representing the model's prediction for what token follows the **last** token in the input sequence.

 In this case, we can see that the model predicts the token `' I'`.

 ### **Step 5:** Add this to the end of the input, re-run

 There are more efficient ways to do this (e.g. where we cache some of the values each time we run our input, so we don't have to do as much calculation each time we generate a new value), but this doesn't matter conceptually right now.

In [14]:
print(f"Sequence so far: {reference_gpt2.to_string(tokens)[0]!r}")

print("Now predicting what comes after the current sequence, one token at a time...")
for i in range(10):
    print(f"{tokens.shape[-1] + 1}th char = {next_char!r}")
    # Define new input sequence, by appending the previously generated token
    tokens = t.cat([tokens, next_token[None, None]], dim=-1)
    # Pass our new sequence through the model, to get new output
    logits = reference_gpt2(tokens)
    # Get the predicted token at the end of our sequence
    next_token = logits[0, -1].argmax(dim=-1)
    # Decode and print the result
    next_char = reference_gpt2.to_string(next_token)


Sequence so far: '<|endoftext|>I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will be able to generate coherent and interesting text!'
Now predicting what comes after the current sequence, one token at a time...
35th char = '\n'
36th char = '\n'
37th char = 'I'
38th char = ' am'
39th char = ' an'
40th char = ' amazing'
41th char = ' aut'
42th char = 'ore'
43th char = 'gressive'
44th char = ','


### Temperature Sampling (Exercise 4)

> ```yaml
> Difficulty: easy-medium
> Time: ~10-15 minutes
> ```

The generation loop above always picks the most likely token (`argmax`). This is called greedy decoding -- it works, but the output tends to be repetitive.

We can add randomness by *sampling* from the distribution instead. A temperature parameter $T$ controls how random the sampling is:

$$
p_i = \frac{e^{z_i / T}}{\sum_{j=1}^{n} e^{z_j / T}}
$$

This is just softmax with the logits divided by $T$ first.

| Temperature | What happens |
|------------|--------|
| $T = 1.0$ | Normal softmax, no change |
| $T < 1.0$ | Distribution gets sharper -- output more predictable |
| $T > 1.0$ | Distribution gets flatter -- output more random |
| $T \to 0$ | Approaches greedy decoding |

Modify the generation loop: divide logits by temperature, then sample with [`t.multinomial(probs, num_samples=1)`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) instead of `argmax`.

In [ ]:
def generate_with_temperature(model, text: str, max_new_tokens: int = 20, temperature: float = 1.0):
    """Generate text using temperature sampling."""
    tokens = model.to_tokens(text).to(device)

    for _ in range(max_new_tokens):
        logits = model(tokens)

        last_logits = ...        # TODO: last position
        scaled_logits = ...      # TODO: divide by temperature
        probs = ...              # TODO: softmax
        next_token = ...         # TODO: sample with t.multinomial

        tokens = t.cat([tokens, next_token[None, None]], dim=-1)

    return model.to_string(tokens[0])

# Try different temperatures!
prompt = "The meaning of life is"
print("=== Temperature 0.1 (nearly greedy) ===")
print(generate_with_temperature(reference_gpt2, prompt, temperature=0.1))

print("\n=== Temperature 0.7 (balanced) ===")
print(generate_with_temperature(reference_gpt2, prompt, temperature=0.7))

print("\n=== Temperature 1.5 (creative) ===")
print(generate_with_temperature(reference_gpt2, prompt, temperature=1.5))

<details><summary>Hint</summary>

The four lines inside the loop:
```python
last_logits = logits[0, -1]
scaled_logits = last_logits / max(temperature, 1e-5)
probs = scaled_logits.softmax(dim=-1)
next_token = t.multinomial(probs, num_samples=1).squeeze()
```
`t.multinomial` samples a token index from the probability distribution. `.squeeze()` removes the extra dimension for concatenation.
</details>

 > ## Key takeaways
 >
 > * Transformer takes in language, predicts next token (for *each* token in a causal way)
 > * We convert language to a sequence of integers with a tokenizer.
 > * We convert integers to vectors with a lookup table.
 > * Output is a vector of logits (one for each input token), we convert to a probability distn with a softmax, and can then convert this to a token (eg taking the largest logit, or sampling).
 > * We append this to the input + run again to generate more text (Jargon: *autoregressive*)
 > * Meta level point: Transformers are sequence operation models, they take in a sequence, do processing in parallel at each position, and use attention to move information between positions!

 In the next notebook (*Lab 2: Building a Transformer from Scratch*), we will open up the black box and implement each of these components (embeddings, attention heads, MLPs, and the full model) from scratch.